# 5.14 · 不平衡分类 / Imbalanced Classification

> **课程定位 / Where this fits**
> 前 13 课大多假设类别大致均衡。但欺诈、罕见病、点击率等**正类只占 1% 甚至 0.1%**。这时准确率彻底失效(全猜负类就 99% 准)。3.11 初探过不平衡数据, 5.1 讲过 PR-AUC; 这一课**系统化**所有武器: 重采样(SMOTE)、类权重、阈值移动、代价敏感、合适的指标。
> When the positive class is rare (fraud, disease), accuracy is useless. The full toolkit: resampling (SMOTE), class weights, threshold shifting, cost-sensitive learning, and the right metrics.

> 💡 **面试相关 / Interview-relevant**
> - "类别不平衡为什么准确率失效 / 用什么指标" ★★★★★
> - "过采样 vs 欠采样 / SMOTE 原理" ★★★★★
> - "class_weight 怎么起作用" ★★★★★
> - "重采样必须在哪一步做(防泄漏)" ★★★★★（只在训练折!）
> - "阈值移动 vs 重采样" ★★★★

---

## 学习目标 / Learning Objectives
1. 为何准确率在不平衡下骗人, 该看什么(PR-AUC/recall, 接 5.1)。
2. **重采样**: 随机过/欠采样, **SMOTE** 原理。
3. **class_weight** 代价敏感学习。
4. **阈值移动**。
5. **防泄漏**: 重采样只能在训练折内(接 3.9/3.12)。

## 目录 / TOC
1. [准确率陷阱 + 指标 ⭐](#1)
2. [💳 数据: 合成信用卡欺诈](#2)
3. [重采样: 过/欠采样 + SMOTE ⭐](#3)
4. [class_weight 代价敏感 ⭐](#4)
5. [阈值移动 + 防泄漏管道 ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 准确率陷阱 + 指标 ⭐ / The Accuracy Trap

正类占 1% 时, "永远预测负类"的废模型准确率 = 99%。**准确率被多数类主导, 完全无意义**。该看:
- **recall(召回)**: 抓到了多少真欺诈(漏报代价高时最重要)。
- **precision**: 报警的有多少是真的。
- **PR-AUC / average precision**(5.1): 不平衡的首选综合指标(ROC-AUC 会过度乐观)。
- **F1 / F-beta**: P/R 的(加权)调和。

**核心思路**: 不平衡不是"换个模型", 而是"换个评估 + 让模型重视少数类"。


<a id="2"></a>
## 2. 数据: 合成信用卡欺诈 / Synthetic Credit-Card Fraud

真 Kaggle 信用卡欺诈集需下载。这里**内联合成**一个 1% 欺诈率的数据集: 正常交易和欺诈交易在几个特征(金额、时间、距离上次交易等)上分布不同, 但**高度重叠**(欺诈本就难抓)。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")

def make_fraud(n=20000, fraud_rate=0.01, seed=0):
    rng = np.random.default_rng(seed)
    n_fraud = int(n * fraud_rate); n_ok = n - n_fraud
    # 正常交易
    ok = np.c_[rng.normal(50, 30, n_ok).clip(1, None),      # amount
               rng.normal(12, 6, n_ok) % 24,                 # hour
               rng.exponential(5, n_ok)]                     # dist_from_last
    # 欺诈: 金额更大、更可能深夜、距离更远, 但重叠
    fr = np.c_[rng.normal(120, 80, n_fraud).clip(1, None),
               (rng.normal(2, 4, n_fraud) % 24),
               rng.exponential(20, n_fraud)]
    X = np.vstack([ok, fr]); y = np.r_[np.zeros(n_ok), np.ones(n_fraud)].astype(int)
    idx = rng.permutation(n)
    return pd.DataFrame(X[idx], columns=["amount","hour","dist_from_last"]), y[idx]

X, y = make_fraud()
print(f"合成欺诈: {X.shape}, 欺诈率 {y.mean():.2%} ({y.sum()} / {len(y)})")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, average_precision_score, f1_score
base = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
pred = base.predict(X_te); proba = base.predict_proba(X_te)[:,1]
print(f"\n朴素逻辑回归(默认阈值0.5):")
print(f"  准确率 {accuracy_score(y_te, pred):.3f} ← 看着很高(被99%正常主导)")
print(f"  recall {recall_score(y_te, pred):.3f} ← 真相: 大量欺诈没抓到!")
print(f"  precision {precision_score(y_te, pred):.3f}  PR-AUC {average_precision_score(y_te, proba):.3f}")


<a id="3"></a>
## 3. 重采样: 过/欠采样 + SMOTE ⭐ / Resampling

平衡训练集的三种做法(用 `imbalanced-learn`):
- **随机欠采样(undersample)**: 丢弃多数类样本。快, 但丢信息。
- **随机过采样(oversample)**: 复制少数类。不丢信息, 但易过拟合(重复点)。
- **SMOTE**(Synthetic Minority Over-sampling): **合成**新少数类样本——在少数类样本与其近邻间**插值**生成新点, 而非简单复制。更平滑, 最常用。


In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

print("各方法重采样后训练集类别分布:")
for name, sampler in [("原始", None),
                      ("随机欠采样", RandomUnderSampler(random_state=0)),
                      ("随机过采样", RandomOverSampler(random_state=0)),
                      ("SMOTE", SMOTE(random_state=0))]:
    if sampler is None:
        Xr, yr = X_tr.values, y_tr
    else:
        Xr, yr = sampler.fit_resample(X_tr, y_tr)
    print(f"  {name:<10} {np.bincount(yr)}")

# SMOTE 插值示意 (2D) / SMOTE interpolation illustration
Xs, ys = SMOTE(random_state=0).fit_resample(X_tr[["amount","dist_from_last"]], y_tr)
fig, ax = plt.subplots(figsize=(7, 4.5))
new_mask = np.arange(len(Xs)) >= len(X_tr)   # 新合成的点
ax.scatter(X_tr["amount"], X_tr["dist_from_last"], c=y_tr, cmap="coolwarm", s=8, alpha=0.4, label="原始")
synth = Xs[new_mask & (ys==1)]
ax.scatter(synth["amount"], synth["dist_from_last"], facecolor="none", edgecolor="green", s=20, label="SMOTE 合成欺诈")
ax.set_xlim(0, 300); ax.set_xlabel("amount"); ax.set_ylabel("dist_from_last"); ax.legend()
ax.set_title("SMOTE: 在少数类近邻间插值合成新样本(绿圈), 而非复制")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. class_weight 代价敏感 ⭐ / Cost-Sensitive Learning

不动数据, 改**损失**: 给少数类的错误**更高权重**。`class_weight='balanced'` 自动按类频率倒数加权——等价于"少数类的每个样本算更多次"。无需重采样、无泄漏风险, 是最省事的首选。


In [ ]:
from sklearn.metrics import precision_recall_curve

results = {}
# 各方法对比 / compare approaches
def evaluate(name, model, Xtr_, ytr_):
    model.fit(Xtr_, ytr_)
    p = model.predict_proba(X_te)[:,1]
    results[name] = (recall_score(y_te, (p>0.5).astype(int)),
                     precision_score(y_te, (p>0.5).astype(int), zero_division=0),
                     average_precision_score(y_te, p))

evaluate("baseline", LogisticRegression(max_iter=1000), X_tr, y_tr)
evaluate("class_weight=balanced", LogisticRegression(max_iter=1000, class_weight="balanced"), X_tr, y_tr)
Xsm, ysm = SMOTE(random_state=0).fit_resample(X_tr, y_tr)
evaluate("SMOTE", LogisticRegression(max_iter=1000), Xsm, ysm)

print(f"{'方法':<24}{'recall':>8}{'precision':>11}{'PR-AUC':>9}")
for k,(r,p,ap) in results.items():
    print(f"{k:<24}{r:>8.3f}{p:>11.3f}{ap:>9.3f}")
print("\nbalanced/SMOTE 大幅提升 recall(抓到更多欺诈), 代价是 precision 下降(更多误报)")
print("⚠️ 注意 PR-AUC 没怎么涨甚至略降: 重采样/加权主要是'移动工作点'(默认阈值下更敢报正类),")
print("   并不必然改善底层的概率排序能力。别迷信 SMOTE——它换的是 recall, 不是排序。")
print("PR-AUC 衡量阈值无关的排序质量, 不平衡下最该看它。")


<a id="5"></a>
## 5. 阈值移动 + 防泄漏管道 ⭐ / Threshold & Leak-Free Pipeline

**阈值移动**: 不重采样不加权, 只把决策阈值从 0.5 **降低**(5.1 讲过)。因为模型概率排序本身可能就不错, 调阈值即可换取 recall。常和 PR 曲线一起选最佳工作点。


In [ ]:
prec, rec, thr = precision_recall_curve(y_te, proba)
f1s = 2*prec*rec/(prec+rec+1e-12)
best_t = thr[np.argmax(f1s[:-1])]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thr, prec[:-1], label="precision"); ax.plot(thr, rec[:-1], label="recall")
ax.plot(thr, f1s[:-1], label="F1", ls="--")
ax.axvline(best_t, color="r", ls=":", label=f"最佳F1阈值={best_t:.2f}")
ax.set_xlabel("阈值"); ax.legend(); ax.set_title("阈值移动: 降阈值换更高 recall, 按 PR/F1 选工作点")
plt.tight_layout(); plt.show()
print(f"默认0.5: recall={recall_score(y_te, proba>0.5):.3f}; 最佳F1阈值{best_t:.2f}: recall={recall_score(y_te, proba>best_t):.3f}")


In [ ]:
# 防泄漏: SMOTE 必须在 CV 的"每个训练折内"做, 不能先采样再划分! / SMOTE inside CV only
from imblearn.pipeline import Pipeline as ImbPipeline   # 注意: 用 imblearn 的 Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 正确: 管道把 SMOTE 关进每个训练折, 验证折保持原始分布
pipe = ImbPipeline([("scale", StandardScaler()),
                    ("smote", SMOTE(random_state=0)),
                    ("clf", LogisticRegression(max_iter=1000))])
cv = StratifiedKFold(5, shuffle=True, random_state=0)
ap_correct = cross_val_score(pipe, X, y, cv=cv, scoring="average_precision")
print(f"✅ 正确(SMOTE 在管道内, 只作用训练折) PR-AUC: {ap_correct.mean():.3f} ± {ap_correct.std():.3f}")
print("❌ 错误做法: 先对全数据 SMOTE 再划分 → 合成样本泄漏进验证折 → 虚高、不可信")
print("规则: 重采样/缩放都只 fit 训练折; 用 imblearn.pipeline.Pipeline 保证这点(接 3.9/3.12)")


<a id="6"></a>
## 6. 小结 / Summary

```
不平衡: 准确率失效(多数主导) → 看 recall / precision / PR-AUC(5.1)
重采样: 随机欠采样(丢信息) / 随机过采样(易过拟合) / SMOTE(近邻插值合成, 最常用)
class_weight='balanced': 改损失给少数类更高权重, 省事无泄漏, 首选
阈值移动: 降阈值换 recall, 按 PR/F1 选工作点
防泄漏 ⭐: 重采样只在训练折! 用 imblearn.pipeline.Pipeline; 先采样再划分=泄漏
```

### 💡 面试速查
1. **准确率失效** → recall / precision / **PR-AUC**(不平衡首选)
2. **SMOTE = 近邻插值合成**少数类(非复制); 欠采样丢信息, 过采样易过拟合
3. **class_weight='balanced'** 改损失加权, 等价代价敏感, 最省事
4. **阈值移动**: 概率排序好时降阈值即可提 recall
5. **重采样只能在训练折内**(imblearn Pipeline), 否则合成样本泄漏到验证集

### 下一节
**5.15 在线学习 + 校准(收官)**——数据流式到达怎么增量训练(partial_fit/SGD)? 以及让概率"可信"的概率校准, 为 Part 5 收尾。
